# 4.14 Seaborn ile Görselleştirme

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/04-matplotlib/14-visualization-with-seaborn.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Visualization with Seaborn

Matplotlib on yıllardır Python'da bilimsel görselleştirmenin merkezinde olsa da en sadık kullanıcıları bile çoğu zaman yetersiz bıraktığını kabul eder. Matplotlib hakkında sık öne çıkan birkaç eleştiri vardır:

Bu sorunlara bir yanıt Seaborn'dur. Seaborn, Matplotlib üzerinde grafik stili ve renk varsayılanları için mantıklı seçenekler sunan, yaygın istatistiksel grafik türleri için basit üst düzey fonksiyonlar tanımlayan ve Pandas'ın sağladığı işlevsellikle bütünleşen bir API sağlar.

Adaletli olmak gerekirse Matplotlib ekibi de değişen ortama uyum sağladı: 4.11 Matplotlib Özelleştirme: Yapılandırma ve Stil Sayfaları bölümünde ele alınan plt.style araçlarını ekledi ve Pandas verisini giderek daha sorunsuz işlemeye başlıyor. Ancak az önce sayılan nedenlerle Seaborn yararlı bir eklenti olmaya devam ediyor.

Gelenek gereği Seaborn genelde sns olarak içe aktarılır. Aşağıdaki hücrede Matplotlib ile birlikte NumPy, Pandas ve Seaborn içe aktarılır; bu bölüm Pyodide'da Pandas + Matplotlib + Seaborn ön yüklemesi kullanır:


```
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set()  # seaborn's method to set its chart style
```


> **Not**
>

## Seaborn Grafiklerini Keşfetme

Seaborn'un ana fikri, istatistiksel veri keşfi için yararlı çeşitli grafik türlerini oluşturmaya yönelik üst düzey komutlar — hatta bazı istatistiksel model uydurmaları — sağlamasıdır.

Seaborn'da kullanılabilir birkaç veri kümesine ve grafik türüne bakalım. Aşağıdakilerin hepsi ham Matplotlib komutlarıyla yapılabilir (Seaborn aslında altta bunu yapar); ancak Seaborn API'si çok daha uygundur.

### Histogramlar, KDE ve Yoğunluklar

İstatistiksel veri görselleştirmede çoğu zaman yalnızca histogram ve değişkenlerin birleşik dağılımlarını çizmek istersiniz. Matplotlib'da bunun nispeten doğrudan yapılabildiğini gördük (aşağıdaki şekil):


In [ ]:
# matplotlib_hist.py
data = np.random.multivariate_normal([0, 0], [[5, 2], [2, 2]], size=2000)
data = pd.DataFrame(data, columns=['x', 'y'])

for col in 'xy':
    plt.hist(data[col], density=True, alpha=0.5)



Görsel çıktı olarak yalnızca histogram yerine, çekirdek yoğunluk tahmini (KDE) ile dağılımın pürüzsüz bir tahminini alabiliriz (4.4 Yoğunluk ve Kontur Grafikleri bölümünde tanıtıldı); Seaborn bunu sns.kdeplot ile yapar (aşağıdaki şekil):


In [ ]:
# kdeplot_1d.py
sns.kdeplot(data=data, shade=True);



kdeplot'a x ve y sütunları geçirirsek bunun yerine birleşik yoğunluğun iki boyutlu görselleştirmesini elde ederiz (aşağıdaki şekil):


In [ ]:
# kdeplot_2d.py
sns.kdeplot(data=data, x='x', y='y');



Birleşik dağılımı ve kenar dağılımlarını birlikte görmek için bu bölümün ilerisinde daha ayrıntılı inceleyeceğimiz sns.jointplot kullanılabilir.

### Çift Grafikler (Pair Plots)

Birleşik grafikleri daha büyük boyutlu veri kümelerine genellediğinizde pair plot'lara (çift grafiklere) ulaşırsınız. Çok boyutlu veride korelasyonları keşfetmek için — tüm değer çiftlerini birbirine karşı çizmek istediğinizde — çok yararlıdırlar.

Bunu iyi bilinen Iris veri kümesiyle göstereceğiz; üç Iris türünün taç yaprak ve çanak yaprak ölçümlerini listeler:


In [ ]:
# iris_load.py
iris = sns.load_dataset("iris")
iris.head()



Örnekler arasındaki çok boyutlu ilişkileri görselleştirmek sns.pairplot çağırmak kadar kolaydır (aşağıdaki şekil):


In [ ]:
# pairplot_iris.py
sns.pairplot(iris, hue='species', height=2.5);



### Yüzeyle ayrılmış histogramlar

Bazen veriyi alt kümelerin histogramlarıyla görmek en iyisidir (aşağıdaki şekil). Seaborn'un FacetGrid'i bunu basitleştirir. Çeşitli gösterge verilerine göre restoran personelinin aldığı bahşiş miktarına bakan bir veri kümesine bakacağız:1

1 Bu bölümde kullanılan restoran personeli verisi çalışanları iki cinsiyete ayırır: kadın ve erkek. Biyolojik cinsiyet ikili değildir; ancak aşağıdaki tartışma ve görselleştirmeler bu veriyle sınırlıdır.


In [ ]:
# tips_load.py
tips = sns.load_dataset('tips')
tips.head()



In [ ]:
# facetgrid_hist.py
tips['tip_pct'] = 100 * tips['tip'] / tips['total_bill']

grid = sns.FacetGrid(tips, row="sex", col="time", margin_titles=True)
grid.map(plt.hist, "tip_pct", bins=np.linspace(0, 40, 15));



Yüzeyle ayrılmış grafik veri kümesine hızlı içgörüler verir: örneğin akşam yemeği saatinde erkek garsonlara ait verinin diğer kategorilere göre çok daha fazla olduğunu ve tipik bahşiş yüzdelerinin yaklaşık %10–20 aralığında olduğunu, her iki uçta aykırı değerler olduğunu görürüz.

### Kategorik Grafikler

Kategorik grafikler bu tür görselleştirme için de yararlı olabilir. Başka bir parametreyle tanımlanan kutular içinde bir parametrenin dağılımını görmenize olanak tanır (aşağıdaki şekil):


In [ ]:
# catplot_box.py
with sns.axes_style(style='ticks'):
    g = sns.catplot(x="day", y="total_bill", hue="sex", data=tips, kind="box")
    g.set_axis_labels("Day", "Total Bill");



### Birleşik Dağılımlar

Daha önce gördüğümüz pair plot'a benzer şekilde farklı veri kümeleri arasındaki birleşik dağılımı ve ilişkili kenar dağılımlarını göstermek için sns.jointplot kullanılabilir (aşağıdaki şekil):


In [ ]:
# jointplot_hex.py
with sns.axes_style('white'):
    sns.jointplot(x="total_bill", y="tip", data=tips, kind='hex')



Birleşik grafik otomatik çekirdek yoğunluk tahmini ve regresyon da yapabilir (aşağıdaki şekil):


In [ ]:
# jointplot_reg.py
sns.jointplot(x="total_bill", y="tip", data=tips, kind='reg');



### Çubuk Grafikleri

Zaman serileri sns.factorplot ile çizilebilir. Aşağıdaki örnekte 3.8 Agregasyon ve Gruplama bölümünde ilk gördüğümüz Gezegenler veri kümesini kullanacağız; sonuç aşağıdaki şekildedir:


In [ ]:
# planets_load.py
planets = sns.load_dataset('planets')
planets.head()



In [ ]:
# catplot_count_year.py
with sns.axes_style('white'):
    g = sns.catplot(x="year", data=planets, aspect=2,
                    kind="count", color='steelblue')
    g.set_xticklabels(step=5)



Bu gezegenlerin keşif yöntemine bakarak daha fazla bilgi edinebiliriz (aşağıdaki şekil):


In [ ]:
# catplot_count_method.py
with sns.axes_style('white'):
    g = sns.catplot(x="year", data=planets, aspect=4.0, kind='count',
                    hue='method', order=range(2001, 2015))
    g.set_ylabels('Number of Planets Discovered')



Seaborn ile çizim hakkında daha fazla bilgi için Seaborn dokümantasyonuna ve özellikle örnek galerisine bakın.

## Örnek: Maraton Bitiş Sürelerini Keşfetme

Burada Seaborn'u bir maratonun bitiş sonuçlarını görselleştirmeye ve anlamaya yardımcı olmak için kullanacağız. Veriyi web kaynaklarından kazıdım, birleştirdim ve tanımlayıcı bilgileri kaldırdım; GitHub'a koydum, oradan indirilebilir (Python ile web kazıma ilgileniyorsanız O'Reilly'den Ryan Mitchell'in Web Scraping with Python kitabını öneririm). Veriyi indirip Pandas'a yükleyerek başlayacağız:2

2 Bu bölümde kullanılan maraton verisi koşucuları iki cinsiyete ayırır: erkek ve kadın. Cinsiyet bir spektrum olsa da aşağıdaki tartışma ve görselleştirmeler veriye bağlı oldukları için bu ikili ayrımı kullanır.


In [ ]:
# url = ('https://raw.githubusercontent.com/jakevdp/'
#        'marathon-data/master/marathon-data.csv')
# !cd data && curl -O {url}



> **Not**
>


In [ ]:
# read_marathon.py
data = pd.read_csv('data/marathon-data.csv')
data.head()



Pandas'ın zaman sütunlarını Python dizileri (object tipi) olarak yüklediğine dikkat edin; bunu DataFrame'in dtypes özniteliğine bakarak görebiliriz:


In [ ]:
# dtypes_before.py
data.dtypes



Zamanlar için bir dönüştürücü sağlayarak bunu düzeltelim:


In [ ]:
# convert_time.py
import datetime

def convert_time(s):
    h, m, s = map(int, s.split(':'))
    return datetime.timedelta(hours=h, minutes=m, seconds=s)

data = pd.read_csv('data/marathon-data.csv',
                   converters={'split':convert_time, 'final':convert_time})
data.head()



In [ ]:
# dtypes_after.py
data.dtypes



Bu, zaman verisini işlemeyi kolaylaştırır. Seaborn çizim yardımcılarımız için ardından süreleri saniye cinsinden veren sütunlar ekleyelim:


In [ ]:
# split_final_sec.py
data['split_sec'] = data['split'].view(int) / 1E9
data['final_sec'] = data['final'].view(int) / 1E9
data.head()



Verinin nasıl göründüğüne dair fikir edinmek için veri üzerinde bir jointplot çizebiliriz; aşağıdaki şekil sonucu gösterir:


In [ ]:
# jointplot_marathon.py
with sns.axes_style('white'):
    g = sns.jointplot(x='split_sec', y='final_sec', data=data, kind='hex')
    g.ax_joint.plot(np.linspace(4000, 16000),
                    np.linspace(8000, 32000), ':k')



Kesikli çizgi, birinin maratonu tamamen sabit tempoda koşsaydı süresinin nerede olacağını gösterir. Dağılımın bunun üstünde olması (beklediğiniz gibi) çoğu insanın maraton boyunca yavaşladığını gösterir. Rekabet koşmuşsanız ikinci yarıda hızlananların — yani yarışı "negatif bölmüş" olanların — sözlükte adının geçtiğini bilirsiniz.

Veride başka bir sütun oluşturalım: bölünme oranı (split_frac), her koşucunun yarışı ne ölçüde negatif veya pozitif böldüğünü ölçer:


In [ ]:
# split_frac.py
data['split_frac'] = 1 - 2 * data['split_sec'] / data['final_sec']
data.head()



Bu bölünme farkı sıfırın altındaysa kişi yarışı o oranda negatif bölmüştür. Bu bölünme oranının dağılım grafiğini çizelim (aşağıdaki şekil):


In [ ]:
# displot_split_frac.py
sns.displot(data['split_frac'], kde=False)
plt.axvline(0, color="k", linestyle="--");



In [ ]:
# count_negative_split.py
sum(data.split_frac < 0)



Yaklaşık 40.000 katılımcıdan yalnızca 250 kişi maratonunu negatif bölmüştür.

Bu bölünme oranı ile diğer değişkenler arasında korelasyon olup olmadığına bakalım. Bunu tüm bu korelasyonların grafiklerini çizen bir PairGrid ile yapacağız (aşağıdaki şekil):


In [ ]:
# pairgrid_marathon.py
g = sns.PairGrid(data, vars=['age', 'split_sec', 'final_sec', 'split_frac'],
                 hue='gender', palette='RdBu_r')
g.map(plt.scatter, alpha=0.8)
g.add_legend();



Bölünme oranı yaşla özellikle korelasyon göstermiyor gibi görünüyor; ancak bitiş süresiyle korelasyon gösteriyor: daha hızlı koşucular maraton sürelerini daha dengeli bölmeye eğilimli. Cinsiyete göre ayrılmış bölünme oranı histogramlarına yakınlaşalım (aşağıdaki şekil):


In [ ]:
# kdeplot_gender.py
sns.kdeplot(data.split_frac[data.gender=='M'], label='men', shade=True)
sns.kdeplot(data.split_frac[data.gender=='W'], label='women', shade=True)
plt.xlabel('split_frac');



İlginç olan, erkekler arasında neredeyse eşit bölmeye yakın koşan çok daha fazla kişi var! Erkekler ve kadınlar arasında neredeyse çift modlu bir dağılım görünüyor. Dağılımları yaş fonksiyonu olarak inceleyerek neler olup bittiğini anlamaya çalışalım.

Dağılımları karşılaştırmanın güzel bir yolu keman grafiği (violin plot) kullanmaktır (aşağıdaki şekil):


In [ ]:
# violinplot_gender.py
sns.violinplot(x="gender", y="split_frac", data=data,
               palette=["lightblue", "lightpink"]);



Biraz daha derine inelim ve bu keman grafiklerini yaşa göre karşılaştıralım (aşağıdaki şekil). Her kişinin on yıllık yaş aralığında olduğunu belirten yeni bir sütun oluşturarak başlayacağız:


In [ ]:
# age_dec.py
data['age_dec'] = data.age.map(lambda age: 10 * (age // 10))
data.head()



In [ ]:
# violinplot_age_dec.py
men = (data.gender == 'M')
women = (data.gender == 'W')

with sns.axes_style(style=None):
    sns.violinplot(x="age_dec", y="split_frac", hue="gender", data=data,
                   split=True, inner="quartile",
                   palette=["lightblue", "lightpink"]);



Erkek ve kadın dağılımlarının nerede farklılaştığını görebiliriz: 20'li–50'li yaşlardaki erkeklerin bölünme dağılımları, aynı yaş grubundaki (veya herhangi bir yaştaki) kadınlara kıyasla düşük bölünmeye doğru belirgin bir aşırı yoğunluk gösteriyor.

Ayrıca şaşırtıcı biçimde 80 yaşındaki kadınlar bölünme sürelerinde herkesi geride bırakıyor gibi görünüyor; ancak bu muhtemelen az sayıda etkisidir, çünkü bu aralıkta yalnızca bir avuç koşucu var:


In [ ]:
# count_age_80.py
(data.age > 80).sum()



Negatif bölen erkeklere dönelim: bunlar kim? Bölünme oranı hızlı bitişle korelasyon gösteriyor mu? Bunu çok kolay çizebiliriz. regplot kullanacağız; veriye otomatik doğrusal regresyon modeli uydurur (aşağıdaki şekil):


In [ ]:
# lmplot_marathon.py
g = sns.lmplot(x='final_sec', y='split_frac', col='gender', data=data,
               markers=".", scatter_kws=dict(color='c'))
g.map(plt.axhline, y=0.0, color="k", ls=":");



Görünüşe göre hem erkekler hem kadınlar arasında hızlı bölenler, yaklaşık 15.000 saniye (yaklaşık 4 saat) içinde bitiren daha hızlı koşuculardır. Bundan yavaş olanların ikinci yarıyı hızlı koşma olasılığı çok daha düşüktür.

### 🧪 Şimdi deneyin

🧪 Seaborn + Pandas
      Küçük bir DataFrame ile sns.scatterplot deneyin — sütun adlarını doğrudan kullanın:
          
      import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({'x': [1, 2, 3, 4], 'y': [1, 4, 2, 3], 'grup': ['A', 'A', 'B', 'B']})
sns.scatterplot(data=df, x='x', y='y', hue='grup')
plt.title('Seaborn scatterplot')
plt.show()

> **Not**
>
